# 01 · ดึงข้อมูลย้อนหลังสำหรับโมเดลทำนายน้ำท่วมหาดใหญ่

Notebook นี้ดึงข้อมูลย้อนหลังจากแหล่งเปิดที่ตรวจสอบแล้ว (ดูรายละเอียดใน `docs/data-sources.md`)
แล้วบันทึกเป็นไฟล์ใน `data/raw/` เพื่อให้ notebook สำรวจข้อมูล (`02_data_exploration.ipynb`) และขั้นเทรนโมเดลใช้ต่อ

| ส่วน | ข้อมูล | แหล่ง | ไฟล์ |
|---|---|---|---|
| A1 | รายชื่อสถานีในลุ่มน้ำอู่ตะเภา | ThaiWater | `stations.parquet` |
| A2 | ระดับน้ำ + ปริมาณน้ำ (โทรมาตร รายชั่วโมง / 10 นาที) | ThaiWater | `waterlevel_tele.parquet` |
| A3 | ฝนรายวันจากสถานีวัดฝน | ThaiWater | `rain_daily_gauge.parquet` |
| A4 | ระดับน้ำ 06:00 รายวัน ทุกสถานีภาคใต้ | กรมชลประทาน ศูนย์อุทกวิทยาที่ 8 | `rid_daily.parquet` |
| A5 | ระดับน้ำสูงสุดรายปี X.44 (2510–ปัจจุบัน) | กรมชลประทาน hydro-8.com | `x44_annual_max.parquet` |
| B1 | ฝน + ความชื้นในดิน ERA5 รายชั่วโมง | Open-Meteo | `era5_hourly.parquet` |
| B2 | ปริมาณน้ำท่าจำลอง GloFAS รายวัน | Open-Meteo Flood API | `glofas_daily.parquet` |
| B3 | ฝนรายวันระยะยาว สนามบินหาดใหญ่ / สงขลา | NOAA GHCN-Daily | `ghcnd_daily.parquet` |
| B4 | ดัชนี ENSO (ONI) | NOAA CPC | `oni.parquet` |
| B5 | พยากรณ์ฝนในอดีต แยกตามวันล่วงหน้า (ECMWF, GFS) | Open-Meteo Previous Runs | `nwp_rain_previous_runs.parquet` |
| C1 | ขอบเขตน้ำท่วมจากดาวเทียม Sentinel-1 พ.ย. 2568 | Copernicus GFM | `gfm_2025/*.tif` |
| C2 | แผนที่น้ำท่วม (flood proxy) 23 พ.ย. 2568 | Sentinel Asia / EOS-RS | `eosrs_flood_2025.parquet` |
| C3 | แบบจำลองความสูงภูมิประเทศ 30 ม. | Copernicus DEM | `dem_glo30.tif` |

**การรันซ้ำ:** แต่ละส่วนจะข้ามถ้ามีไฟล์อยู่แล้ว (ตั้ง `FORCE = True` เพื่อโหลดใหม่)
และทุก HTTP response ถูก cache ไว้ใน `data/cache/` การรันครั้งแรกใช้เวลาประมาณ 20–40 นาที

In [2]:
import datetime

import pandas

from mojito import config, sources

FORCE = False  # True = ดึงใหม่ทั้งหมด

TODAY = datetime.date.today()
RAW = config.RAW_DIR
RAW.mkdir(parents=True, exist_ok=True)
session = sources.make_session(cache=True)

pandas.set_option("display.max_columns", 20)


def save_once(name, fetch):
    """Run `fetch` and save its DataFrame, unless the file already exists."""
    path = RAW / name
    if path.exists() and not FORCE:
        frame = pandas.read_parquet(path)
        print(f"skip  {name}: มีอยู่แล้ว {len(frame):,} แถว")
        return frame
    frame = fetch()
    frame.to_parquet(path, index=False)
    print(f"saved {name}: {len(frame):,} แถว")
    return frame

## A · ข้อมูลโทรมาตรน้ำของไทย

### A1 · รายชื่อสถานีในลุ่มน้ำ
ดึงจาก feed ล่าสุดของ ThaiWater (`waterlevel_load`, `rain_24h`) แล้วกรองเฉพาะสถานีที่อยู่ในกรอบลุ่มน้ำอู่ตะเภา
ได้พิกัด ระดับตลิ่ง (`min_bank`) และหน่วยงานเจ้าของสถานี

In [3]:
stations = save_once("stations.parquet", lambda: sources.thaiwater_stations(session))
stations.groupby(["kind", "agency"]).size().rename("จำนวนสถานี").to_frame()

skip  stations.parquet: มีอยู่แล้ว 81 แถว


จำนวนสถานี
kind       agency              
rain       DISASTER           4
           DWR               32
           FOP                2
           HII               11
           RID               15
           TMD                4
waterlevel HII                7
           RID                6

In [4]:
key_ids = [s["id"] for s in config.WATERLEVEL_STATIONS.values()]
stations[stations["station_id"].isin(key_ids)].sort_values("lat")

,station_id,code,name,kind,agency,lat,lon,min_bank,ground_level,qmax
2,2585,X.173A,บ้านม่วงก็อง,waterlevel,RID,6.796630,100.442337,16.13,3.748,272.899994
10,860,SLA002,คลองหอยโข่ง,waterlevel,HII,6.931234,100.439850,9.34,-1.013,NaN
29,860,SLA002,คลองหอยโข่ง,rain,HII,6.931234,100.439850,NaN,NaN,NaN
9,2589,X.90,บ้านบางศาลา,waterlevel,RID,6.931240,100.439857,9.34,-0.881,777.799988
5,2590,X.174,บ้านคลองหวะ,waterlevel,RID,6.986370,100.481857,8.88,0.000,190.100006
8,2591,X.44,บ้านหาดใหญ่ใน,waterlevel,RID,7.002180,100.456062,7.15,-2.204,481.000000
6,858,SLA007,บางกล่ำ,waterlevel,HII,7.079418,100.427880,1.59,-1.739,NaN
30,858,SLA007,บางกล่ำ,rain,HII,7.079418,100.427880,NaN,NaN,NaN
1,1109526,ONE037,สะพานข้ามคลองอู่ตะเภา,waterlevel,HII,7.134997,100.453210,0.84,-4.515,NaN
34,1109526,ONE037,สะพานข้ามคลองอู่ตะเภา,rain,HII,7.134997,100.453210,NaN,NaN,NaN


### A2 · ระดับน้ำและปริมาณน้ำจากโทรมาตร
`waterlevel_yearly_graph` ให้ข้อมูลทั้งปีในการเรียกครั้งเดียว ความถี่ขึ้นกับสถานี
(สถานีกรมชลฯ รายชั่วโมง, สถานี HII ทุก 10 นาที)

- สถานีกรมชลฯ (X.44, X.90, X.173A, X.174) มีข้อมูลตั้งแต่ **2020**
- สถานี HII (SLA002, SLA007) มีตั้งแต่ **2012**, SLA005 ตั้งแต่ 2014, ONE037/038 ตั้งแต่ 2022

In [4]:
WATERLEVEL_YEARS = range(2012, TODAY.year + 1)

waterlevel = save_once(
    "waterlevel_tele.parquet",
    lambda: sources.thaiwater_waterlevel(session, config.WATERLEVEL_STATIONS, WATERLEVEL_YEARS),
)
waterlevel.groupby("station_code")["timestamp"].agg(["min", "max", "count"])

skip  waterlevel_tele.parquet: มีอยู่แล้ว 2,090,520 แถว


,min,max,count
station_code,,,
ONE037,2022-11-18 11:10:00+07:00,2026-10-04 16:50:00+07:00,149709
ONE038,2022-11-18 15:20:00+07:00,2026-10-04 17:00:00+07:00,179946
SLA002,2012-04-05 14:50:00+07:00,2026-10-04 16:50:00+07:00,556036
SLA005,2013-08-05 21:00:00+07:00,2026-10-04 16:50:00+07:00,458269
SLA007,2012-09-06 12:50:00+07:00,2026-10-04 16:50:00+07:00,544611
X.173A,2020-02-13 01:00:00+07:00,2026-10-04 16:00:00+07:00,50542
X.174,2020-02-13 01:00:00+07:00,2026-10-04 16:00:00+07:00,50433
X.44,2012-11-30 06:00:00+07:00,2026-10-04 16:00:00+07:00,50547
X.90,2012-11-30 06:00:00+07:00,2026-10-04 16:00:00+07:00,50427


### A3 · ฝนรายวันจากสถานีวัดฝน
สถานีหลักที่กำหนดไว้ใน `config.RAIN_STATIONS` และสถานีวัดฝนทุกสถานีที่อยู่ในเขตตัวเมืองหาดใหญ่
เรียก `rain_yearly_graph` ก่อนเพื่อดูว่าเดือนไหนมีข้อมูล แล้วค่อยดึงรายวันเฉพาะเดือนนั้น

In [5]:
rain_stations = {code: dict(info) for code, info in config.RAIN_STATIONS.items()}

min_lon, min_lat, max_lon, max_lat = config.CITY_BBOX
city_gauges = stations[
    (stations["kind"] == "rain")
    & stations["lon"].between(min_lon, max_lon)
    & stations["lat"].between(min_lat, max_lat)
]
known_ids = {s["id"] for s in rain_stations.values()}
for gauge in city_gauges.itertuples():
    if gauge.station_id not in known_ids:
        rain_stations[gauge.code or str(gauge.station_id)] = {
            "id": gauge.station_id, "name": gauge.name, "agency": gauge.agency,
        }

print(f"สถานีวัดฝนที่จะดึง {len(rain_stations)} สถานี:", ", ".join(rain_stations))

rain_gauge = save_once(
    "rain_daily_gauge.parquet",
    lambda: sources.thaiwater_rain_daily(session, rain_stations, range(2012, TODAY.year + 1)),
)
rain_gauge.groupby("station_code")["date"].agg(["min", "max", "count"])

สถานีวัดฝนที่จะดึง 21 สถานี: 48569, 48568, SLA001, NPCT, MOU421, STN0580, STN2213, ridtele_STN05, ridtele_STN07, STN0710, G09006-TA200304, STN2212, SLA002, SLA007, STN2062, ridtele_STN01, ridtele_STN02, ridtele_TSL24, ridtele_TSL27, ridtele_TSL40, 48571
skip  rain_daily_gauge.parquet: มีอยู่แล้ว 24,339 แถว


,min,max,count
station_code,,,
48568,2017-08-04,2026-09-29,2641
48569,2017-08-04,2026-09-29,2633
48571,2017-08-04,2026-09-29,2652
MOU421,2026-09-19,2026-09-28,9
NPCT,2014-05-20,2026-09-29,4220
SLA001,2020-09-09,2026-09-29,2052
SLA002,2020-09-11,2026-09-29,1891
SLA007,2020-09-09,2026-09-29,2054
STN0580,2014-11-10,2026-09-29,2663


### A4 · ระดับน้ำรายวัน (06:00) จากกรมชลประทาน ศูนย์อุทกวิทยาที่ 8
API ส่งข้อมูลครั้งละ 7 วันของทุกสถานีในภาคใต้ (77 สถานี) ระบุวันเป็นปี พ.ศ.
มีข้อมูลตั้งแต่กลางปี 2560 (2017) พร้อมระดับตลิ่ง (`bank_level`) และความจุลำน้ำ (`qmax`) ของแต่ละสถานี

In [6]:
rid = save_once(
    "rid_daily.parquet",
    lambda: sources.rid_daily(session, datetime.date(2017, 6, 1), TODAY),
)
rid[rid["basin"] == "ทะเลสาบสงขลา"].groupby("station_code")["date"].agg(["min", "max", "count"])

skip  rid_daily.parquet: มีอยู่แล้ว 183,870 แถว


,min,max,count
station_code,,,
X.112,2020-06-19,2026-10-04,1967
X.170,2017-06-01,2026-10-04,3217
X.173A,2017-06-01,2026-10-04,3219
X.174,2017-06-01,2026-10-04,3219
X.240,2020-06-19,2026-10-04,2074
X.265,2017-06-01,2026-10-04,3217
X.267,2020-06-19,2026-10-04,2066
X.282,2020-06-19,2026-10-04,2060
X.292,2024-11-12,2026-10-04,692


### A5 · ระดับน้ำสูงสุดรายปี สถานี X.44 (หาดใหญ่ใน)
อ่านจากกราฟ PDF ของศูนย์อุทกวิทยาภาคใต้ ครอบคลุมปีน้ำ 2510–ปัจจุบัน
เป็นข้อมูลเดียวที่ย้อนไปถึงน้ำท่วมใหญ่ปี 2543 และ 2553 (ระดับเริ่มท่วมพื้นที่ลุ่มต่ำ 7.40 ม.รทก.)

In [7]:
annual_max = save_once("x44_annual_max.parquet", lambda: sources.hydro8_annual_max(session, "X44"))
annual_max.sort_values("max_stage_msl", ascending=False).head(8)

skip  x44_annual_max.parquet: มีอยู่แล้ว 61 แถว


,water_year,max_stage_msl
59,2025,9.97
44,2010,9.10
34,2000,8.77
22,1988,8.45
39,2005,7.68
7,1973,7.27
58,2024,7.26
45,2011,7.05


## B · สภาพอากาศและปริมาณน้ำท่า

### B1 · ERA5 reanalysis (ฝน + ความชื้นในดิน) รายชั่วโมง
6 จุดกริด 0.25° คลุมลุ่มน้ำ ตั้งแต่ปี 1990 ระบุ `models=era5` ทุกช่วงปีให้ข้อมูลมาจากแหล่งเดียวกัน
(ข้อมูลล่าช้าประมาณ 5 วัน)

In [8]:
era5 = save_once(
    "era5_hourly.parquet",
    lambda: sources.era5_hourly(
        session, config.ERA5_POINTS, 1990, TODAY - datetime.timedelta(days=7)
    ),
)
era5.groupby(["lat", "lon"])["timestamp"].agg(["min", "max", "count"])

skip  era5_hourly.parquet: มีอยู่แล้ว 1,932,336 แถว


min                       max   count
lat  lon                                                               
6.75 100.25 1990-01-01 00:00:00+07:00 2026-09-27 23:00:00+07:00  322056
     100.50 1990-01-01 00:00:00+07:00 2026-09-27 23:00:00+07:00  322056
7.00 100.25 1990-01-01 00:00:00+07:00 2026-09-27 23:00:00+07:00  322056
     100.50 1990-01-01 00:00:00+07:00 2026-09-27 23:00:00+07:00  322056
7.25 100.25 1990-01-01 00:00:00+07:00 2026-09-27 23:00:00+07:00  322056
     100.50 1990-01-01 00:00:00+07:00 2026-09-27 23:00:00+07:00  322056

### B2 · GloFAS river discharge รายวัน
แบบจำลองปริมาณน้ำท่าในลำน้ำระดับโลกของ Copernicus ที่ cell คลองอู่ตะเภาสายหลัก (ใกล้ตัวเมือง)
และ cell ต้นน้ำ ใช้เป็นทั้ง feature และตัวเทียบกับปริมาณน้ำที่วัดจริง
(ขอตั้งแต่ปี 1984 แต่ Open-Meteo มีข้อมูลจุดนี้เริ่มปี 1997 ซึ่งยังครอบเหตุการณ์ 2543 และ 2553)

In [9]:
glofas = save_once(
    "glofas_daily.parquet",
    lambda: sources.glofas_daily(session, config.GLOFAS_POINTS, datetime.date(1984, 1, 1), TODAY),
)
glofas.groupby("point").agg(
    cell_lat=("cell_lat", "first"), cell_lon=("cell_lon", "first"),
    start=("date", "min"), end=("date", "max"), max_discharge=("river_discharge", "max"),
)

skip  glofas_daily.parquet: มีอยู่แล้ว 21,738 แถว


,cell_lat,cell_lon,start,end,max_discharge
point,,,,,
utapao_main,7.025002,100.42502,1997-01-01,2026-10-04,1580.25
utapao_upstream,6.875000,100.42502,1997-01-01,2026-10-04,1103.08


### B3 · NOAA GHCN-Daily ฝนรายวันสถานีกรมอุตุฯ
สนามบินหาดใหญ่ (48569) ตั้งแต่ปี 1973 และสงขลา (48568) ตั้งแต่ปี 1943
เป็นข้อมูลฝนสถานีวัดจริงที่ยาวที่สุดที่เปิดให้ดาวน์โหลด (หน่วยในไฟล์ต้นทางเป็น 0.1 มม. แปลงเป็น มม. แล้ว)

In [10]:
ghcnd = save_once("ghcnd_daily.parquet", lambda: sources.ghcnd_daily(session, config.GHCND_STATIONS))
ghcnd.dropna(subset=["rain_mm"]).groupby("station_code")["date"].agg(["min", "max", "count"])

skip  ghcnd_daily.parquet: มีอยู่แล้ว 49,161 แถว


,min,max,count
station_code,,,
48568,1943-01-01,2025-08-24,25597
48569,1973-06-20,2025-08-24,15080


### B4 · ดัชนี ONI (El Niño / La Niña)
ค่าบวก ≥ 0.5 = El Niño, ค่าลบ ≤ -0.5 = La Niña ใช้เป็น feature ระดับฤดูกาล

In [11]:
oni = save_once("oni.parquet", lambda: sources.oni(session))
oni.tail(6)

skip  oni.parquet: มีอยู่แล้ว 920 แถว


,date,season,oni
914,2026-03-01,FMA,0.11
915,2026-04-01,MAM,0.46
916,2026-05-01,AMJ,0.95
917,2026-06-01,MJJ,1.39
918,2026-07-01,JJA,1.80
919,2026-08-01,JAS,2.16


### B5 · พยากรณ์ฝนในอดีต (Open-Meteo Previous Runs)
ค่าฝนที่แบบจำลอง **ECMWF IFS 0.25°** และ **GFS** เคยพยากรณ์ไว้ล่วงหน้า 1–6 วันสำหรับแต่ละชั่วโมง
ใช้ back-test ว่าถ้ามีพยากรณ์ฝน ณ วันนั้นจริงๆ โมเดลระดับน้ำจะดีขึ้นแค่ไหน — มีข้อมูลตั้งแต่ประมาณ มี.ค. 2024 เท่านั้น

In [12]:
def fetch_previous_runs():
    return pandas.concat([
        sources.nwp_rain_previous_runs(session, config.ERA5_POINTS, model, sources.PREVIOUS_RUNS_START, TODAY)
        for model in ("ecmwf_ifs025", "gfs_seamless")
    ], ignore_index=True)


nwp_rain = save_once("nwp_rain_previous_runs.parquet", fetch_previous_runs)
nwp_rain.groupby(["model", "lead"])["timestamp"].agg(["min", "max"]).unstack("lead").iloc[:, [0, 5, 6, 11]]

saved nwp_rain_previous_runs.parquet: 1,637,370 แถว


min                            \
lead                                 1                         6   
model                                                              
ecmwf_ifs025 2024-03-01 00:00:00+07:00 2024-03-01 00:00:00+07:00   
gfs_seamless 2024-03-01 00:00:00+07:00 2024-03-01 00:00:00+07:00   

                                   max                            
lead                                 1                         6  
model                                                             
ecmwf_ifs025 2026-10-04 23:00:00+07:00 2026-10-04 23:00:00+07:00  
gfs_seamless 2026-10-04 23:00:00+07:00 2026-10-04 23:00:00+07:00

## C · ข้อมูลเชิงพื้นที่

### C1 · ขอบเขตน้ำท่วมจริงจาก Sentinel-1 (Copernicus GFM) เหตุการณ์ พ.ย. 2568
เป็น label เชิงพื้นที่สำหรับโมเดล heatmap ความละเอียด 20 ม. ค่าในไฟล์: 0 = แห้ง, 1 = น้ำท่วม, 255 = ไม่มีข้อมูล
(ตัดเฉพาะกรอบลุ่มน้ำ ไม่ต้อง login)

In [13]:
GFM_DIR = RAW / "gfm_2025"
gfm_index = save_once(
    "gfm_2025_index.parquet",
    lambda: sources.gfm_flood_extent(GFM_DIR, config.BASIN_BBOX, "2025-11-15", "2025-12-05"),
)
gfm_index.sort_values("datetime")

skip  gfm_2025_index.parquet: มีอยู่แล้ว 12 แถว


,scene,datetime,path,valid_pct,flooded_km2
10,ENSEMBLE_FLOOD_20251123T230309_VV_AS020M_E051N...,2025-11-23 23:03:09+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251123T2303...,70.0,3.47
11,ENSEMBLE_FLOOD_20251123T230309_VV_AS020M_E048N...,2025-11-23 23:03:09+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251123T2303...,75.7,0.11
8,ENSEMBLE_FLOOD_20251129T113521_VV_AS020M_E051N...,2025-11-29 11:35:21+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251129T1135...,90.5,15.83
9,ENSEMBLE_FLOOD_20251129T113521_VV_AS020M_E048N...,2025-11-29 11:35:21+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251129T1135...,100.0,0.77
6,ENSEMBLE_FLOOD_20251129T230152_VV_AS020M_E051N...,2025-11-29 23:01:52+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251129T2301...,31.7,10.01
7,ENSEMBLE_FLOOD_20251129T230152_VV_AS020M_E048N...,2025-11-29 23:01:52+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251129T2301...,40.3,0.19
5,ENSEMBLE_FLOOD_20251129T230221_VV_OC020M_E033N...,2025-11-29 23:02:21+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251129T2302...,38.8,0.33
4,ENSEMBLE_FLOOD_20251205T113401_VV_OC020M_E033N...,2025-12-05 11:34:01+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251205T1134...,18.2,0.28
2,ENSEMBLE_FLOOD_20251205T113426_VV_AS020M_E051N...,2025-12-05 11:34:26+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251205T1134...,65.6,2.08
3,ENSEMBLE_FLOOD_20251205T113426_VV_AS020M_E048N...,2025-12-05 11:34:26+00:00,data/raw/gfm_2025/ENSEMBLE_FLOOD_20251205T1134...,84.7,0.06


### C2 · แผนที่น้ำท่วม (flood proxy) จาก Sentinel Asia / EOS-RS
GFM จำแนกน้ำจากความสว่างของภาพเรดาร์ ซึ่งจับน้ำท่วมระหว่างอาคารในเขตเมืองได้ไม่ดี
EOS-RS (Earth Observatory of Singapore) ใช้ change detection ของ Sentinel-1 ก่อน/หลังน้ำท่วม
จึงเห็นน้ำท่วมในตัวเมืองได้มากกว่า เผยแพร่เป็น polygon ผ่าน Sentinel Asia (ภาพวันที่ 23 พ.ย. 2568)

In [14]:
EOSRS_PATH = RAW / "eosrs_flood_2025.parquet"
if EOSRS_PATH.exists() and not FORCE:
    print("skip  eosrs_flood_2025.parquet: มีอยู่แล้ว")
else:
    eosrs = sources.sentinel_asia_flood_proxy(session, config.BASIN_BBOX)
    eosrs.to_parquet(EOSRS_PATH)
    print(f"saved eosrs_flood_2025.parquet: {len(eosrs):,} polygon")

skip  eosrs_flood_2025.parquet: มีอยู่แล้ว


### C3 · Copernicus GLO-30 DEM
ความสูงภูมิประเทศ 30 ม. ใช้คำนวณ HAND / ความชัน / ทิศทางการไหล ในขั้น feature engineering

In [15]:
DEM_PATH = RAW / "dem_glo30.tif"
if DEM_PATH.exists() and not FORCE:
    print("skip  dem_glo30.tif: มีอยู่แล้ว")
else:
    dem = sources.copernicus_dem(DEM_PATH, config.BASIN_BBOX)
    print("saved dem_glo30.tif", dem.shape)

skip  dem_glo30.tif: มีอยู่แล้ว


## ข้อมูลที่ยังไม่ได้ดึงในรอบนี้

| ข้อมูล | เหตุผล | ทำต่อเมื่อ |
|---|---|---|
| GPM IMERG / JAXA GSMaP (ฝนจากดาวเทียม 0.1°) | ต้องสมัคร NASA Earthdata / JAXA | ทำโมเดล ConvLSTM แบบกริด |
| GISTDA พื้นที่น้ำท่วมซ้ำซาก 2554–2566 | ต้องขอ API key | ทำ label เชิงพื้นที่เพิ่ม |
| FABDEM (DEM ตัดอาคาร/ต้นไม้ออก) | ไฟล์ 556 MB | ทำ feature HAND ในเขตเมือง |
| HII CSV ระดับน้ำ 10 นาที | ส่วนใหญ่ซ้ำกับ A2 | ต้องการความละเอียดสูงกว่ารายชั่วโมง |
| ระดับน้ำ/ฝนก่อนปี 2012 รายวัน | ต้องทำหนังสือขอจากกรมชลประทาน | ต้องการเทรนกับเหตุการณ์ 2543 / 2553 |

## สรุปข้อมูลที่ได้

In [16]:
def summarize(name, frame, time_column):
    times = pandas.to_datetime(frame[time_column])
    value_columns = frame.select_dtypes("number").columns
    return {
        "ไฟล์": name,
        "แถว": len(frame),
        "เริ่ม": times.min().date(),
        "ถึง": times.max().date(),
        "% ค่าว่าง (ตัวเลข)": round(100 * frame[value_columns].isna().mean().mean(), 1),
    }


pandas.DataFrame([
    summarize("waterlevel_tele", waterlevel, "timestamp"),
    summarize("rain_daily_gauge", rain_gauge, "date"),
    summarize("rid_daily", rid, "date"),
    summarize("x44_annual_max", annual_max.assign(d=pandas.to_datetime(annual_max["water_year"].astype(str))), "d"),
    summarize("era5_hourly", era5, "timestamp"),
    summarize("glofas_daily", glofas, "date"),
    summarize("ghcnd_daily", ghcnd, "date"),
    summarize("oni", oni, "date"),
    summarize("nwp_rain_previous_runs", nwp_rain, "timestamp"),
])

,ไฟล์,แถว,เริ่ม,ถึง,% ค่าว่าง (ตัวเลข)
0,waterlevel_tele,2090520,2012-04-05,2026-10-04,45.2
1,rain_daily_gauge,24339,2014-05-20,2026-09-29,0.0
2,rid_daily,183870,2017-06-01,2026-10-04,21.2
3,x44_annual_max,61,1966-01-01,2026-01-01,2.5
4,era5_hourly,1932336,1990-01-01,2026-09-27,0.0
5,glofas_daily,21738,1997-01-01,2026-10-04,0.0
6,ghcnd_daily,49161,1943-01-01,2025-08-24,14.3
7,oni,920,1950-01-01,2026-08-01,0.0
8,nwp_rain_previous_runs,1637370,2024-03-01,2026-10-04,0.0


### ตรวจความถูกต้องกับค่าอ้างอิงที่ทราบ
ค่าเหล่านี้ยืนยันจากแหล่งต้นทางตอนสำรวจข้อมูล ถ้า assert ไม่ผ่านแปลว่าการ parse ผิดพลาด

In [17]:
x44 = waterlevel[waterlevel["station_code"] == "X.44"]
peak = x44.loc[x44["waterlevel_msl"].idxmax()]
assert abs(peak["waterlevel_msl"] - 9.97) < 0.05 and peak["timestamp"].date() == datetime.date(2025, 11, 25), peak

annual = annual_max.set_index("water_year")["max_stage_msl"]
assert annual[2000] == 8.77 and annual[2010] == 9.10 and annual[2025] == 9.97

hatyai_era5 = era5[(era5["lat"] == 7.0) & (era5["lon"] == 100.5)]
era5_daily = hatyai_era5.set_index("timestamp")["precipitation"].resample("D").sum()
assert abs(era5_daily["2025-11"].max() - 240) < 10, era5_daily["2025-11"].max()

ghcnd_rain = ghcnd[ghcnd["station_code"] == "48569"].set_index("date")["rain_mm"]
assert abs(ghcnd_rain[datetime.date(2010, 11, 1)] - 240.8) < 0.1

main = glofas[glofas["point"] == "utapao_main"].set_index("date")["river_discharge"]
assert abs(main[(main.index >= datetime.date(2025, 11, 1)) & (main.index <= datetime.date(2025, 11, 30))].max() - 1580) < 50

print("ผ่านทุกการตรวจ ✓")

ผ่านทุกการตรวจ ✓
